In [1]:
from pathlib import Path
from datetime import date

import duckdb
con = duckdb.connect()
bronze_path = "../data/bronze/2026-07-11/products.parquet" 




In [2]:
con.execute(f"SELECT COUNT(*) FROM '{bronze_path}'").fetchone()

(15145,)

In [3]:
KEEP_COLUMNS = [
    "code", "product_name", "generic_name", "created_datetime",
    "last_modified_datetime", "countries_tags", "brands",
    "categories_tags", "categories", "main_category",
    "completeness", "quantity", "product_quantity",
]
columns_sql = ", ".join(f'"{col}"' for col in KEEP_COLUMNS)

result = con.execute(f"""
    SELECT COUNT(*), COUNT(DISTINCT code)
    FROM (SELECT {columns_sql} FROM '{bronze_path}')
""").fetchone()
print(result)

(15145, 15145)


In [7]:
print(columns_sql)

"code", "product_name", "generic_name", "created_datetime", "last_modified_datetime", "countries_tags", "brands", "categories_tags", "categories", "main_category", "completeness", "quantity", "product_quantity"


In [4]:
result = con.execute(f"""
    SELECT COUNT(*)
    FROM (SELECT {columns_sql} FROM '{bronze_path}')
    WHERE code IS NOT NULL
      AND NOT (product_name IS NULL AND generic_name IS NULL)
""").fetchone()
print(result)

(12167,)


In [5]:
result = con.execute(f"""
    SELECT code, last_modified_datetime, rn
    FROM (
        SELECT
            {columns_sql},
            ROW_NUMBER() OVER (
                PARTITION BY code
                ORDER BY last_modified_datetime DESC
            ) AS rn
        FROM '{bronze_path}'
        WHERE code IS NOT NULL
          AND NOT (product_name IS NULL AND generic_name IS NULL)
    )
    ORDER BY code
    LIMIT 10
""").fetchall()
print(result)

[('00001007', datetime.datetime(2025, 1, 3, 19, 50, 31, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), 1), ('0000105604159', datetime.datetime(2025, 5, 27, 4, 21, 39, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), 1), ('0000109616677', datetime.datetime(2024, 10, 3, 0, 14, 37, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), 1), ('0000116043502', datetime.datetime(2025, 6, 3, 12, 12, 1, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), 1), ('0000205733847', datetime.datetime(2024, 10, 3, 0, 15, 31, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), 1), ('0000338004206', datetime.datetime(2023, 5, 9, 7, 15, 14, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), 1), ('00005680', datetime.datetime(2024, 10, 3, 0, 16, 31, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), 1), ('0000605132457', datetime.datetime(2025, 1, 21, 17, 17, 52, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), 1), ('00011823', datetime.datetime(2024, 10, 3, 0, 14, 46, tzinfo=<DstTzInfo 'Asia/Manila' PST

In [6]:
result = con.execute(f"""
    SELECT code, last_modified_datetime AT TIME ZONE 'UTC' AS utc_time
    FROM '{bronze_path}'
    ORDER BY code
    LIMIT 3
""").fetchall()
print(result)

[('00001007', datetime.datetime(2025, 1, 3, 11, 50, 31)), ('0000105604159', datetime.datetime(2025, 5, 26, 20, 21, 39)), ('0000109616677', datetime.datetime(2024, 10, 2, 16, 14, 37))]


In [8]:
result = con.execute(f"""
    SELECT code, last_modified_datetime
    FROM (
        SELECT code, last_modified_datetime AT TIME ZONE 'UTC' AS last_modified_datetime
        FROM '{bronze_path}'
        ORDER BY last_modified_datetime DESC
        LIMIT 3
    )
""").fetchall()
print(result)

[('3616661245994', datetime.datetime(2026, 3, 25, 23, 25, 17)), ('3760260842723', datetime.datetime(2026, 3, 25, 23, 10, 52)), ('3760260845137', datetime.datetime(2026, 3, 25, 23, 8, 23))]
